# 01totals: national monthly and yearly totals

**What it does.** Reads the six national tables from the KORAIL statistics portal, turns them into tidy tables, checks them against each other, against the station tables from `00load`, and against KORAIL's published totals, and saves them to `data/clean`.

- Riders by train type, monthly, January 2016 to December 2023 (KORAIL and SRT).
- Riders by line and train type, yearly, 2004 to 2024.
- Scheduled train runs and train km by train type, 2016 to 2024.
- Operating revenue, monthly, 2016 to 2024.
- Railway accidents and service disruptions, monthly, 2016 to 2024.

**Inputs:** `data/raw/railstat/` (tables B3 to B8), `data/clean/station_year_ktx_srt.csv` (from `00load`).
**Outputs:** `data/clean/riders_month.csv`, `line_year.csv`, `runs_year.csv`, `revenue_month.csv`, `disruptions_month.csv`.

**Data credit:** Korea Railroad Corporation (KORAIL), Korean Railway Statistics portal (railstat.korail.com). Free to use, no restrictions on scope of use.

## 0. Settings

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("data/raw/railstat")  # <- folder with the portal downloads
CLEAN = Path("data/clean")  # <- folder for the tidy tables

FILE_RIDERS = RAW / "연도별_여객수송(인원)_20261029131453.csv"  # <- B3
FILE_LINE_1624 = RAW / "선별_여객_수송실적(인원)_20261029131725.csv"  # <- B4
FILE_LINE_0415 = RAW / "선별_여객_수송실적(인원)_20261029131840.csv"  # <- B5
FILE_RUNS = RAW / "열차종별_운행횟수_20261029132049.csv"  # <- B6
FILE_DISRUPT = RAW / "철도사고·운행장애_발생현황_20261029132225.csv"  # <- B7
FILE_REVENUE = RAW / "연도별_운수수입실적_20261029132436.csv"  # <- B8
FILE_STATIONS = CLEAN / "station_year_ktx_srt.csv"  # <- from 00load

# KORAIL's published yearly intercity riders (info.korail.com) and SR riders from 00load
PUBLISHED_KORAIL = {2004: 111_214_000, 2023: 136_693_000, 2024: 144_805_000}  # <- check figures
SR_2023 = 26_412_000  # <- SR boardings 2023, station table in 00load (rounded)

pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 3.1.0


## 1. Helpers

The portal's wide exports have several header rows (one per level of the column hierarchy) and then data rows. `read_wide` finds the header rows by their repeated first cell, returns the headers as a small table with one row per data column, and the values as numbers. A dash means zero or not applicable and becomes empty.

In [2]:
def to_num(s):
    s = s.astype("str").str.replace(",", "", regex=False)
    s = s.replace({"-": None, "": None, "nan": None})
    return pd.to_numeric(s, errors="coerce")


def read_wide(path, n_key_cols=1):
    raw = pd.read_csv(path, encoding="utf-8-sig", header=None, dtype=str)
    n_head = int((raw[0] == raw.iloc[0, 0]).sum())
    head = raw.iloc[:n_head, n_key_cols:].T.reset_index(drop=True)
    head.columns = [f"h{i}" for i in range(n_head)]
    keys = raw.iloc[n_head:, :n_key_cols].reset_index(drop=True)
    values = raw.iloc[n_head:, n_key_cols:].apply(to_num).reset_index(drop=True)
    values.columns = range(values.shape[1])
    return head, keys, values


def split_periods(keys, values):
    period = keys[0]
    is_month = period.str.startswith("M")
    is_year = period.str.startswith("Y")
    monthly = values[is_month].copy()
    monthly.index = pd.to_datetime(period[is_month].str[1:7], format="%Y%m")
    yearly = values[is_year].copy()
    yearly.index = period[is_year].str[1:5].astype(int)
    return monthly, yearly


def check_months(monthly, yearly):
    sums = monthly.groupby(monthly.index.year).sum(min_count=1)
    diff = (sums - yearly.loc[sums.index]).abs().max().max()
    print("months:", monthly.index.min().strftime("%Y-%m"), "to", monthly.index.max().strftime("%Y-%m"), "|", len(monthly))
    print("months missing:", 12 * len(yearly) - len(monthly))
    print("largest gap between monthly sums and yearly rows:", diff)

## 2. Riders by train type, monthly, 2016 to 2023 (B3)

Riders (boardings) are split into season tickets (정기), ordinary tickets on the main network (일반구간) and other sections (기타). Each train type is the sum of its columns across the three groups. KTX includes the "other sections" high speed column, which makes KTX match the station table exactly. SRT here is 1 to 3% higher than the station table in 2017 to 2019 and equal from 2020; the source does not say why.

In [3]:
head, keys, values = read_wide(FILE_RIDERS)
print(head.to_string())

                 h0                       h1                 h2                h3
0     001 합계(A)=B+I                       소계                 소계                소계
1         002 정기(B)                       소계                 소계                소계
2         002 정기(B)              002001 일반철도     002001001 고속철도  002001001001 KTX
3         002 정기(B)              002001 일반철도     002001001 고속철도  002001001002 SRT
4         002 정기(B)              002001 일반철도      002001002 새마을                소계
5         002 정기(B)              002001 일반철도      002001003 무궁화                소계
6         002 정기(B)              002001 일반철도       002001004 통근                소계
7   003 비정기계(I)=G+H                       소계                 소계                소계
8   003 비정기계(I)=G+H  003001 일반구간계(G)=C+D+E+F                 소계                소계
9   003 비정기계(I)=G+H  003001 일반구간계(G)=C+D+E+F  003001001 고속철도(C)  003001001001 KTX
10  003 비정기계(I)=G+H  003001 일반구간계(G)=C+D+E+F  003001001 고속철도(C)  003001001002 SRT
11  003 비정기계(I)=

In [4]:
label = head.fillna("").agg(" / ".join, axis=1)

TYPE_COLS = {}
TYPE_COLS["total"] = [i for i, s in enumerate(label) if "001 합계" in s]
TYPE_COLS["KTX"] = [i for i, s in enumerate(label) if "KTX" in s or "003002001 고속철도" in s]
TYPE_COLS["SRT"] = [i for i, s in enumerate(label) if "SRT" in s]
TYPE_COLS["Saemaul"] = [i for i, s in enumerate(label) if "새마을" in s]
TYPE_COLS["Mugunghwa"] = [i for i, s in enumerate(label) if "무궁화" in s]
TYPE_COLS["Commuter"] = [i for i, s in enumerate(label) if "통근" in s]
for k, v in TYPE_COLS.items():
    print(k, v)

total [0]
KTX [2, 9, 15]
SRT [3, 10]
Saemaul [4, 11, 16]
Mugunghwa [5, 12, 17]
Commuter [6, 13, 18]


In [5]:
monthly, yearly = split_periods(keys, values)
check_months(monthly, yearly)

riders_m = pd.DataFrame(index=monthly.index)
for k, cols in TYPE_COLS.items():
    riders_m[k] = monthly[cols].sum(axis=1, min_count=1)
riders_m.index.name = "month"

parts = riders_m.drop(columns="total").sum(axis=1)
print("largest gap between train types and total (riders):", (parts - riders_m["total"]).abs().max())

months: 2016-01 to 2023-12 | 96
months missing: 0
largest gap between monthly sums and yearly rows: 0.0
largest gap between train types and total (riders): 0.0


In [6]:
riders_y = riders_m.groupby(riders_m.index.year).sum()
stations = pd.read_csv(FILE_STATIONS, encoding="utf-8-sig")
st_y = stations.groupby(["year", "operator"])["board"].sum().unstack()

cmp = pd.DataFrame(index=riders_y.index)
cmp["KTX_B3_m"] = riders_y["KTX"] / 1e6
cmp["KTX_stations_m"] = st_y["KTX"] / 1e6
cmp["SRT_B3_m"] = riders_y["SRT"] / 1e6
cmp["SRT_stations_m"] = st_y["SRT"] / 1e6
cmp["total_m"] = riders_y["total"] / 1e6
print(cmp.round(2))
print()
print("2023 total = KORAIL published + SR:", round(riders_y.loc[2023, "total"] / 1e6, 2), "=", round(PUBLISHED_KORAIL[2023] / 1e6, 2), "+", round(SR_2023 / 1e6, 2))

       KTX_B3_m  KTX_stations_m  SRT_B3_m  SRT_stations_m  total_m
month                                                             
2016      64.62           64.62      1.01            1.01   136.54
2017      59.83           59.83     19.71           19.47   147.56
2018      67.04           67.04     22.69           21.96   157.47
2019      70.91           70.91     23.97           23.33   163.50
2020      43.56           43.56     17.15           17.15   102.38
2021      50.52           50.52     19.56           19.56   113.10
2022      70.80           70.80     24.10           24.10   145.51
2023      84.01           84.01     26.41           26.41   163.11

2023 total = KORAIL published + SR: 163.11 = 136.69 + 26.41


## 3. Riders by line and train type, yearly, 2004 to 2024 (B4 and B5)

The two periods come from differently shaped tables with different line codes, so lines are matched by name. A rider is counted once on each line used, so the line figures add up to more than the network total (the 합계 row is the network total). From 2016 the network total includes SR, so it is checked against KORAIL's published figure plus SR's station total.

In [7]:
head4, keys4, values4 = read_wide(FILE_LINE_1624)
head4["year"] = head4["h0"].str[-4:].astype(int)
head4["type"] = "other"
head4.loc[head4["h1"].str.contains("합계"), "type"] = "total"
head4.loc[head4["h2"].str.contains("KTX") | head4["h3"].str.contains("KTX"), "type"] = "KTX"
head4.loc[head4["h2"].str.contains("SRT"), "type"] = "SRT"
head4.loc[head4["h2"].str.contains("새마을"), "type"] = "Saemaul"
head4.loc[head4["h2"].str.contains("무궁화"), "type"] = "Mugunghwa"
head4.loc[head4["h2"].str.contains("통근"), "type"] = "Commuter"
print(head4.groupby("type").size().to_dict())

long4 = values4.copy()
long4["line"] = keys4[0].str.split(" ", n=1).str[1].values
long4 = long4.melt(id_vars="line", var_name="col", value_name="riders")
long4["year"] = long4["col"].map(head4["year"])
long4["type"] = long4["col"].map(head4["type"])
line4 = long4.groupby(["year", "line", "type"])["riders"].sum(min_count=1).reset_index()
print(line4["year"].min(), "to", line4["year"].max(), "|", line4["line"].nunique(), "lines")

{'Commuter': 23, 'KTX': 45, 'Mugunghwa': 43, 'SRT': 18, 'Saemaul': 45, 'other': 7, 'total': 9}
2016 to 2024 | 52 lines


In [8]:
b5 = pd.read_csv(FILE_LINE_0415, encoding="utf-8-sig", dtype=str)
b5 = b5[b5.iloc[:, 2] == "소계"]
b5["line"] = b5.iloc[:, 0].str.split(" ", n=1).str[1]
b5["type"] = b5.iloc[:, 1].str.split(" ", n=1).str[1].map({"합계": "total", "고속철도": "KTX", "새마을": "Saemaul", "무궁화": "Mugunghwa", "통근열차": "Commuter"})
year_cols = [c for c in b5.columns if c.startswith("Y")]
line5 = b5.melt(id_vars=["line", "type"], value_vars=year_cols, var_name="period", value_name="riders")
line5["year"] = line5["period"].str[1:5].astype(int)
line5["riders"] = to_num(line5["riders"])
line5 = line5[["year", "line", "type", "riders"]]
print(line5["year"].min(), "to", line5["year"].max(), "|", line5["line"].nunique(), "lines")

2004 to 2015 | 55 lines


In [9]:
line_year = pd.concat([line5, line4], ignore_index=True)
line_year = line_year.sort_values(["line", "type", "year"]).reset_index(drop=True)

net = line_year[(line_year["line"] == "합계") & (line_year["type"] == "total")].set_index("year")["riders"] / 1e6
net_cmp = pd.DataFrame({"network_m": net})
net_cmp["B3_total_m"] = riders_y["total"] / 1e6
net_cmp["published_KORAIL_m"] = pd.Series(PUBLISHED_KORAIL) / 1e6
net_cmp["SR_stations_m"] = st_y["SRT"] / 1e6
net_cmp["published_plus_SR_m"] = net_cmp["published_KORAIL_m"] + net_cmp["SR_stations_m"].fillna(0)
print(net_cmp.round(2).to_string())
print()
only5 = sorted(set(line5["line"]) - set(line4["line"]))
only4 = sorted(set(line4["line"]) - set(line5["line"]))
print("lines only before 2016:", only5)
print("lines only from 2016:", only4)

      network_m  B3_total_m  published_KORAIL_m  SR_stations_m  published_plus_SR_m
year                                                                               
2004     111.21         NaN              111.21            NaN               111.21
2005     115.00         NaN                 NaN            NaN                  NaN
2006     114.33         NaN                 NaN            NaN                  NaN
2007     110.63         NaN                 NaN            NaN                  NaN
2008     113.10         NaN                 NaN            NaN                  NaN
2009     107.73         NaN                 NaN            NaN                  NaN
2010     112.09         NaN                 NaN            NaN                  NaN
2011     121.77         NaN                 NaN            NaN                  NaN
2012     125.82         NaN                 NaN            NaN                  NaN
2013     132.03         NaN                 NaN            NaN              

In [10]:
top = line_year[(line_year["type"] == "KTX") & (line_year["year"].isin([2004, 2015, 2019, 2024]))]
top = top.pivot_table(index="line", columns="year", values="riders", aggfunc="sum") / 1e6
print("KTX riders by line, millions:")
print(top.dropna(how="all").sort_values(2024, ascending=False).round(2).head(12))

KTX riders by line, millions:
year    2004   2015   2019   2024
line                             
합계     19.88  60.53  70.91  89.89
경부고속선   1.49  21.00  28.57  38.16
경부선    16.60  25.89  24.75  27.53
전라선     0.00   1.49   2.92   3.56
호남선     1.32   2.10   2.35   2.91
강릉선      NaN    NaN   2.44   2.61
광주송북연    NaN    NaN   2.00   2.27
경전선     0.00   1.24   1.71   2.11
경원선      NaN    NaN   0.93   1.84
중앙선      NaN    NaN   0.34   1.79
경부3선     NaN    NaN   1.08   1.63
동해선     0.00   0.65   1.20   1.38


## 4. Scheduled train runs and train km, 2016 to 2024 (B6)

The two value columns are KORAIL (labelled KTX in the export) and SR (labelled SRT). The unit is not stated in the export. The figures behave like **one-way trains per day** and **train km per day**: KORAIL high speed has about 240 runs a day averaging about 390 km, which matches Seoul to Busan, and SRT has about 120 a day. Treat them as daily scheduled service, not as yearly totals.

In [11]:
head6, keys6, values6 = read_wide(FILE_RUNS, n_key_cols=2)
head6["year"] = head6["h0"].str[-4:].astype(int)
head6["measure"] = head6["h1"].map(lambda s: "runs_per_day" if "횟수" in s else "train_km_per_day")
head6["operator"] = head6["h2"].map(lambda s: "KORAIL" if "KTX" in s else "SR")

long6 = values6.copy()
long6["group"] = keys6[0].str.split(" ", n=1).str[1].values
long6["train_type"] = keys6[1].str.split(" ", n=1).str[-1].values
long6 = long6.melt(id_vars=["group", "train_type"], var_name="col", value_name="value")
for c in ["year", "measure", "operator"]:
    long6[c] = long6["col"].map(head6[c])
runs = long6.pivot_table(index=["year", "operator", "group", "train_type"], columns="measure", values="value", aggfunc="sum").reset_index()
runs.columns.name = None
runs = runs.dropna(subset=["runs_per_day"])
runs["avg_km_per_run"] = runs["train_km_per_day"] / runs["runs_per_day"]

hs = runs[runs["train_type"].isin(["고속철도", "소계"]) & (runs["group"] == "여객")]
print(hs.pivot_table(index="year", columns=["operator", "train_type"], values=["runs_per_day", "avg_km_per_run"]).round(0))

           avg_km_per_run                      runs_per_day                      
operator           KORAIL            SR              KORAIL             SR       
train_type           고속철도     소계   고속철도     소계         고속철도      소계   고속철도     소계
year                                                                             
2016                392.0  101.0  368.0  368.0        237.0  2997.0  122.0  122.0
2017                392.0  101.0  373.0  373.0        239.0  2995.0  120.0  120.0
2018                365.0  105.0  373.0  373.0        275.0  3210.0  120.0  120.0
2019                366.0  105.0  374.0  374.0        275.0  3188.0  121.0  121.0
2020                367.0  109.0  373.0  373.0        275.0  3084.0  120.0  120.0
2021                354.0  106.0  370.0  370.0        305.0  3103.0  121.0  121.0
2022                354.0  104.0  370.0  370.0        306.0  3215.0  121.0  121.0
2023                355.0  109.0  374.0  374.0        321.0  3237.0  132.0  132.0
2024            

## 5. Operating revenue, monthly, 2016 to 2024 (B8)

In won. Passenger fares for KORAIL and SR, freight, and other income (including the public service obligation payment, PSO). SR revenue is empty until December 2016, when SRT started. **The source also has no SR revenue for 2021**, so 2021 totals are too low by roughly SR's yearly revenue (about 0.5 to 0.6 trillion won in the years around it).

In [12]:
head8, keys8, values8 = read_wide(FILE_REVENUE)
label8 = head8.fillna("").agg(" / ".join, axis=1)
print(label8.to_string())
monthly8, yearly8 = split_periods(keys8, values8)
check_months(monthly8, yearly8)

0                        001 총계(A+B+C+D) / 소계 / 소계
1                   002 여객(A) / 002001 한국철도공사 / 소계
2                      002 여객(A) / 002002 ㈜SR / 소계
3                              004 화물(C) / 소계 / 소계
4                         005 기타영업잡수익(D) / 소계 / 소계
5    005 기타영업잡수익(D) / 005001 여객 / 005001001 한국철도공사
6       005 기타영업잡수익(D) / 005001 여객 / 005001002 ㈜SR
7                  005 기타영업잡수익(D) / 005002 화물 / 소계
8                005 기타영업잡수익(D) / 005004 기타수송 / 소계
9                 005 기타영업잡수익(D) / 005005 PSO / 소계
months: 2016-01 to 2024-12 | 108
months missing: 0
largest gap between monthly sums and yearly rows: 0.0


In [13]:
REV_COLS = {"total": 0, "pax_korail": 1, "pax_sr": 2, "freight": 3, "other": 4, "other_pax_korail": 5, "other_pax_sr": 6, "other_freight": 7, "other_misc": 8, "pso": 9}  # <- column order as printed above
revenue_m = monthly8.rename(columns={v: k for k, v in REV_COLS.items()})
revenue_m.index.name = "month"
rev_y = revenue_m.groupby(revenue_m.index.year).sum(min_count=1) / 1e12
print("trillion won:")
print(rev_y[["total", "pax_korail", "pax_sr", "freight", "pso"]].round(3))
print("months with no SR revenue:", revenue_m.index[revenue_m["pax_sr"].isna()].strftime("%Y-%m").tolist())

trillion won:
       total  pax_korail  pax_sr  freight    pso
month                                           
2016   4.172       3.455   0.029    0.291  0.351
2017   4.351       3.151   0.562    0.294  0.296
2018   4.632       3.341   0.621    0.283  0.324
2019   4.711       3.383   0.645    0.265  0.353
2020   3.242       2.143   0.456    0.243  0.353
2021   3.074       2.398     NaN    0.250  0.380
2022   4.419       3.112   0.620    0.241  0.384
2023   5.013       3.630   0.674    0.231  0.398
2024   5.226       3.868   0.688    0.220  0.361
months with no SR revenue: ['2016-01', '2016-02', '2016-03', '2016-04', '2016-05', '2016-06', '2016-07', '2016-08', '2016-09', '2016-10', '2016-11', '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06', '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12']


## 6. Railway accidents and service disruptions, monthly, 2016 to 2024 (B7)

Counts of events. A railway accident (철도사고) is a collision, derailment or level crossing accident. A service disruption (운행장애) is an event that stops or delays trains without an accident, such as a vehicle fault, a signal fault or a power failure. There is no published punctuality figure, so this table is the closest public measure of reliability.

In [14]:
head7, keys7, values7 = read_wide(FILE_DISRUPT)
label7 = head7.fillna("").agg(" / ".join, axis=1)
print(label7.to_string())
monthly7, yearly7 = split_periods(keys7, values7)
check_months(monthly7, yearly7)

0                              13 총계 / 소계 / 소계
1                            11 철도사고 / 소계 / 소계
2                     11 철도사고 / 1101 열차사고 / 소계
3            11 철도사고 / 1101 열차사고 / 110101 열차충돌
4            11 철도사고 / 1101 열차사고 / 110102 열차탈선
5                      11 철도사고 / 1102 건널목 / 소계
6                            12 운행장애 / 소계 / 소계
7                     12 운행장애 / 1201 위험사건 / 소계
8                     12 운행장애 / 1202 차량탈선 / 소계
9                     12 운행장애 / 1205 열차분리 / 소계
10                    12 운행장애 / 1206 차량구름 / 소계
11                    12 운행장애 / 1207 규정위반 / 소계
12                    12 운행장애 / 1208 선로장애 / 소계
13                    12 운행장애 / 1209 급전장애 / 소계
14                    12 운행장애 / 1210 신호장애 / 소계
15                    12 운행장애 / 1211 차량고장 / 소계
16    12 운행장애 / 1211 차량고장 / 121101 고속차량(KTX산천)
17    12 운행장애 / 1211 차량고장 / 12110101 고속차량(SRT)
18             12 운행장애 / 1211 차량고장 / 121102 전기
19             12 운행장애 / 1211 차량고장 / 121103 디젤
20             12 운행장애 / 1211 차량고장 / 121104 동차
21           

In [15]:
EVENT_COLS = {"total": 0, "accidents": 1, "disruptions": 6, "dis_track": 12, "dis_power": 13, "dis_signal": 14, "dis_vehicle": 15, "dis_vehicle_ktx": 16, "dis_vehicle_srt": 17}  # <- column order as printed above
events_m = monthly7[list(EVENT_COLS.values())].fillna(0).astype(int)
events_m.columns = list(EVENT_COLS.keys())
events_m.index.name = "month"
print("accidents plus disruptions equals total every month:", bool((events_m["accidents"] + events_m["disruptions"] == events_m["total"]).all()))
print(events_m.groupby(events_m.index.year).sum())

accidents plus disruptions equals total every month: True
       total  accidents  disruptions  dis_track  dis_power  dis_signal  dis_vehicle  dis_vehicle_ktx  dis_vehicle_srt
month                                                                                                                
2016     221         16          205          1          6          11          108               36                2
2017     192         13          179          2          7          12           90               10               10
2018     142         10          132          7          2          12          103               29               17
2019     315         20          295          4          5          25          169               60               13
2020     155          9          146          0          6          11           98               43                8
2021     107         18           89          3          3           7           56               25                

## 7. Save

In [16]:
riders_m.to_csv(CLEAN / "riders_month.csv", encoding="utf-8-sig")
line_year.to_csv(CLEAN / "line_year.csv", index=False, encoding="utf-8-sig")
runs.to_csv(CLEAN / "runs_year.csv", index=False, encoding="utf-8-sig")
revenue_m.to_csv(CLEAN / "revenue_month.csv", encoding="utf-8-sig")
events_m.to_csv(CLEAN / "disruptions_month.csv", encoding="utf-8-sig")
for f in sorted(CLEAN.glob("*.csv")):
    print(f.name, f.stat().st_size // 1024, "KB")

disruptions_month.csv 3 KB
ktx_station_month.csv 254 KB
line_year.csv 150 KB
revenue_month.csv 13 KB
riders_month.csv 5 KB
runs_year.csv 10 KB
station_year_all.csv 263 KB
station_year_ktx_srt.csv 115 KB


___
**END**